# Rule prevalence changes

## Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.stats.multitest import multipletests

ROOT = next(
    folder for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (folder / "result_summary" / "data_helper.py").exists()
)
sys.path[:0] = [
    str(ROOT / "result_summary"),
    str(ROOT / "result_summary" / "differential_rules"),
]

import data_helper as dh
import differential_stats as ds
import differential_vis as dv

warnings.filterwarnings("ignore")

RULE_MAX_ITEMS = 2
KIND = None
METRIC = "Lift"
METRIC_REFERENCE = 1
SCORE_COLUMN = "Pathological score"
STAGES = ["Control", "Mild", "Severe"]
ORGANS = ["Colon", "Duodenum"]
STAGE_PAIRS = [("Control", "Mild"), ("Control", "Severe"), ("Mild", "Severe")]

MIN_PRESENT = 3
MIN_ELIGIBLE = 3
MIN_CELLS = 20
N_PERMUTATIONS = 5_000
FDR_CUTOFF = 0.05
MIN_GAP = None
MIN_DETAIL_GAP = 0.20
TOP_TRENDS = 15
TOP_STATES = 3
MAX_DETAILS = 15
MIN_ORGAN_STAGES = 2
USE_CELL_ELIGIBILITY = True

TEX_RULE_EXPORTS = {
    ("Duodenum", "Pathological score", "Fibroblast -> Muscle"): "tex_method_fibroblast_muscle_path.pdf",
    ("Duodenum", "Clinical score", "Fibroblast -> Muscle"): "tex_method_fibroblast_muscle_clinical.pdf",
    ("Colon", "Pathological score", "Goblet -> Muscle"): "tex_method_goblet_muscle_forward.pdf",
    ("Colon", "Pathological score", "Muscle -> Goblet"): "tex_method_goblet_muscle_reverse.pdf",
    ("Duodenum", "Pathological score", "Macrophage -> Muscle"): "tex_new_macrophage_muscle.pdf",
    ("Duodenum", "Pathological score", "CD4T -> Goblet"): "tex_new_duodenum_cd4_goblet_path.pdf",
    ("Duodenum", "Clinical score", "CD4T -> Goblet"): "tex_new_duodenum_cd4_goblet_clinical.pdf",
    ("Colon", "Pathological score", "CD4T -> Goblet"): "tex_new_colon_cd4_goblet.pdf",
    ("Colon", "Pathological score", "Goblet -> CD4T"): "tex_method_direction_goblet_cd4.pdf",
    ("Colon", "Pathological score", "Goblet -> SMV"): "tex_new_goblet_smv.pdf",
    ("Colon", "Pathological score", "Fibroblast -> Plasma"): "tex_new_fibroblast_plasma.pdf",
    ("Duodenum", "Pathological score", "Paneth -> Mast"): "tex_new_paneth_mast.pdf",
    ("Duodenum", "Pathological score", "Paneth -> Plasma_CD38"): "tex_new_paneth_plasma_cd38.pdf",
    ("Duodenum", "Pathological score", "Goblet -> Mesenchymal_VIM"): "tex_new_goblet_mesenchymal_vim.pdf",
    ("Duodenum", "Pathological score", "CD8T -> Goblet"): "tex_paper_duodenum_cd8_goblet.pdf",
}
TEX_ORGAN_EXPORTS = {
    ("Clinical score", "Goblet -> Endothelial"): "tex_method_organ_goblet_endothelial.pdf",
    ("Clinical score", "Goblet -> Muscle"): "tex_new_organ_goblet_muscle.pdf",
    ("Clinical score", "Fibroblast -> Goblet"): "tex_new_organ_fibroblast_goblet.pdf",
}

## Data

In [ ]:
df_cells, df_fovs, df_biopsy = dh.load_spatial_data()
df_results = dh.load_results(rule_max_items=RULE_MAX_ITEMS, kind=KIND)

## Logic

In [ ]:
def cell_share_matrix(cells):
    counts = cells.groupby(["fov", "cell type"]).size().unstack(fill_value=0)
    return counts.div(counts.sum(axis=1), axis=0).T

In [ ]:
def chance_of(values, weights, observed, seed=0):
    rng = np.random.default_rng(seed)
    shuffled = np.column_stack([
        rng.permutation(weights) for _ in range(N_PERMUTATIONS)
    ])
    null = values @ shuffled
    extreme = np.abs(null) >= np.abs(observed)[:, None] - 1e-12
    return (extreme.sum(axis=1) + 1) / (N_PERMUTATIONS + 1)

In [ ]:
def group_values(matrix, fovs):
    return matrix.reindex(columns=fovs, fill_value=0).to_numpy(float)

In [ ]:
def compare_groups(matrix, fovs_a, fovs_b, label_a, label_b):
    a, b = group_values(matrix, fovs_a), group_values(matrix, fovs_b)
    keep = ((a != 0).sum(1) + (b != 0).sum(1)) >= MIN_PRESENT
    a, b, names = a[keep], b[keep], matrix.index[keep]
    net_a, net_b = a.mean(1), b.mean(1)
    effect = net_a - net_b
    weights = np.r_[
        np.full(a.shape[1], 1 / a.shape[1]),
        np.full(b.shape[1], -1 / b.shape[1]),
    ]
    p_value = chance_of(np.hstack([a, b]), weights, effect)
    return pd.DataFrame({
        f"net_{label_a}": net_a, f"net_{label_b}": net_b,
        "effect_size": effect,
        f"n_attract_{label_a}": (a == 1).sum(1),
        f"n_avoid_{label_a}": (a == -1).sum(1),
        f"n_attract_{label_b}": (b == 1).sum(1),
        f"n_avoid_{label_b}": (b == -1).sum(1),
        "p_value": p_value,
        "fdr": multipletests(p_value, method="fdr_bh")[1],
    }, index=names).sort_values("effect_size", key=abs, ascending=False)

In [ ]:
def mean_metrics_by_group(rules, fov_to_group, metrics=("Lift", "Confidence")):
    table = rules.copy()
    table["Group"] = table["FOV"].map(fov_to_group)
    table["Lift"] = np.log(table["Lift"].clip(lower=1e-6))
    means = table.groupby(["Clean_Rule", "Group"])[list(metrics)].mean()
    means["Lift"] = np.exp(means["Lift"])
    return means.unstack("Group")

In [ ]:
def report_table(comparison, label_a, label_b, metrics=None, top_n=15):
    top = comparison.head(top_n)
    table = pd.DataFrame({
        f"{label_a} net %": (100 * top[f"net_{label_a}"]).round(1),
        f"{label_b} net %": (100 * top[f"net_{label_b}"]).round(1),
        "gap (pp)": (100 * top["effect_size"]).abs().round(1),
        "more in": np.where(top["effect_size"] > 0, label_a, label_b),
        f"{label_a} FOVs + / -": top[f"n_attract_{label_a}"].astype(str) + " / " + top[f"n_avoid_{label_a}"].astype(str),
        f"{label_b} FOVs + / -": top[f"n_attract_{label_b}"].astype(str) + " / " + top[f"n_avoid_{label_b}"].astype(str),
        "FDR": top["fdr"].round(4),
        "passed": np.where(top["fdr"] <= FDR_CUTOFF, "yes", "no"),
    })
    if metrics is not None:
        means = metrics.sort_index(axis=1).loc[:, (slice(None), [label_a, label_b])]
        means.columns = [f"mean {metric} {group}" for metric, group in means.columns]
        table = table.join(means.round(2))
    return table

In [ ]:
def fovs_for(organ=None, stage=None, score=SCORE_COLUMN):
    keep = pd.Series(True, index=df_fovs.index)
    if organ is not None:
        keep &= df_fovs["Organ"] == organ
    if stage is not None:
        keep &= df_fovs[score] == stage
    return df_fovs.loc[keep, "FOV"].tolist()

In [ ]:
def stage_nets(matrix, groups):
    return pd.DataFrame({
        stage: group_values(matrix, fovs).mean(axis=1)
        for stage, fovs in groups.items()
    })

In [ ]:
def compare_stages(matrix, groups):
    nets = stage_nets(matrix, groups)
    fovs = [fov for stage_fovs in groups.values() for fov in stage_fovs]
    values = group_values(matrix, fovs)
    keep = (values != 0).sum(1) >= MIN_PRESENT
    values, nets = values[keep], nets.loc[keep]
    ranks = np.concatenate([
        np.full(len(stage_fovs), rank, float)
        for rank, stage_fovs in enumerate(groups.values())
    ])
    centered = ranks - ranks.mean()
    slope = values @ (centered / np.sum(centered ** 2))
    p_value = chance_of(values, centered / np.sum(centered ** 2), slope)
    steps = np.diff(nets.to_numpy(float), axis=1)
    result = nets.add_prefix("net_")
    result["slope"] = slope
    result["steady"] = (steps >= 0).all(1) | (steps <= 0).all(1)
    result["p_value"] = p_value
    result["fdr"] = multipletests(p_value, method="fdr_bh")[1]
    return result.sort_values("slope", key=abs, ascending=False)

In [ ]:
def trend_table(trend, top_n=8):
    moved = trend[trend["fdr"] <= FDR_CUTOFF]
    rows = pd.concat([
        moved[moved["slope"] > 0].head(top_n),
        moved[moved["slope"] < 0].head(top_n),
    ])
    table = (100 * rows[[f"net_{stage}" for stage in STAGES]]).round(0)
    table.columns = [f"{stage} net %" for stage in STAGES]
    table["pp per stage"] = (100 * rows["slope"]).round(0)
    table["direction"] = np.where(rows["slope"] > 0, "climbing", "fading")
    table["same way every step"] = rows["steady"]
    table["FDR"] = rows["fdr"].round(4)
    return table

In [ ]:
def eligible_compare(fovs_a, fovs_b, label_a, label_b):
    result = ds.compare(
        eligible_states, fovs_a, fovs_b, label_a, label_b,
        min_eligible=MIN_ELIGIBLE,
        min_present=MIN_PRESENT,
        n_permutations=N_PERMUTATIONS,
    )
    ds.correct_together({"comparison": result})
    return result

In [ ]:
def eligible_trend(groups):
    return ds.trend(
        eligible_states, groups, STAGES,
        min_eligible=MIN_ELIGIBLE,
        min_present=MIN_PRESENT,
        n_permutations=N_PERMUTATIONS,
    )

In [ ]:
def strongest_trends(trend):
    passed = trend[trend["fdr"] <= FDR_CUTOFF]
    rising = passed[passed["slope"] > 0].nlargest(TOP_STATES, "slope").index.tolist()
    fading = passed[passed["slope"] < 0].nsmallest(TOP_STATES, "slope").index.tolist()
    order = [rule for pair in zip(rising, fading) for rule in pair]
    order += rising[len(fading):] + fading[len(rising):]
    return passed.loc[order]

In [ ]:
def rank_rules(ordered=None, pairs=None):
    fdr, change = {}, {}
    if ordered is not None:
        fdr["ordered"], change["ordered"] = ordered["fdr"], ordered["slope"].abs()
    for label, result in (pairs or {}).items():
        fdr[str(label)], change[str(label)] = result["fdr"], result["effect_size"].abs()
    return pd.DataFrame(fdr).assign(
        best_fdr=lambda x: x.min(axis=1),
        largest_change=pd.DataFrame(change).max(axis=1),
    ).sort_values(["best_fdr", "largest_change"], ascending=[True, False])

In [ ]:
def interesting_rules(ranking):
    keep = (ranking["best_fdr"] <= FDR_CUTOFF) & (ranking["largest_change"] >= MIN_DETAIL_GAP)
    return ranking.loc[keep].head(MAX_DETAILS)

In [ ]:
def show_score_results(score):
    output = {}
    for organ in ORGANS:
        ordered, pairs = ds.severity_screen(
            eligible_states, df_fovs, organ, score, STAGES,
            min_eligible=MIN_ELIGIBLE,
            min_present=MIN_PRESENT,
            n_permutations=N_PERMUTATIONS,
        )
        selected = interesting_rules(rank_rules(ordered, pairs)).copy()
        columns = [f"n_eligible_{stage}" for stage in STAGES]
        selected["smallest eligible group"] = ordered.reindex(selected.index)[columns].min(1)
        display(selected.round(4))
        tests, gaps = {(organ, score): ordered}, {(organ, score): pairs}
        for rule in selected.index:
            spec = {"rule": rule, "organ": organ, "score": score}
            dv.plot_rule_states(all_states, eligibility, df_fovs, [spec], STAGES,
                                heading=f"{organ} {rule.replace(' -> ', ' → ')} rule states",
                                trend_results=tests, pair_results=gaps)
            dv.plot_rule_and_cell_changes(
                all_states, eligibility, df_cells, df_fovs, [spec], STAGES,
                heading=f"{organ} {rule.replace(' -> ', ' → ')} beside cell abundance",
                trend_results=tests, pair_results=gaps,
                save=TEX_RULE_EXPORTS.get((organ, score, rule)),
            )
            dv.plot_rule_metric(rule_matrix, eligibility, df_fovs, spec, STAGES,
                                metric=METRIC, reference=METRIC_REFERENCE)
        output[organ] = {"trend": ordered, "pairs": pairs, "selected": selected}
    return output

In [ ]:
def organ_stage_screen(score):
    results = {
        stage: ds.compare(
            eligible_states,
            fovs_for("Colon", stage, score),
            fovs_for("Duodenum", stage, score),
            "Colon", "Duodenum",
            min_eligible=MIN_ELIGIBLE,
            min_present=MIN_PRESENT,
            n_permutations=N_PERMUTATIONS,
        )
        for stage in STAGES
    }
    ds.correct_together(results)
    return results, rank_rules(pairs=results)

In [ ]:
def show_organ_results(score):
    results, ranking = organ_stage_screen(score)
    ranking["significant stages"] = pd.DataFrame({
        stage: result["fdr"] <= FDR_CUTOFF
        for stage, result in results.items()
    }).sum(1)
    selected = interesting_rules(
        ranking[ranking["significant stages"] >= MIN_ORGAN_STAGES]
    )
    display(selected.round(4))
    for rule in selected.index:
        dv.plot_organ_rule_context(
            all_states, eligibility, df_cells, df_fovs, rule,
            group_col=score, stages=STAGES, stage_results=results,
            heading=f"Stage-matched {rule.replace(' -> ', ' → ')} by organ",
            save=TEX_ORGAN_EXPORTS.get((score, rule)),
        )
        for organ in ORGANS:
            spec = {"rule": rule, "organ": organ, "score": score}
            dv.plot_rule_metric(rule_matrix, eligibility, df_fovs, spec, STAGES,
                                metric=METRIC, reference=METRIC_REFERENCE)
    return {"tests": results, "selected": selected}

In [ ]:
rule_matrix = dh.prepare_rules(df_results)
all_states, eligibility, eligible_states = ds.state_tables(
    rule_matrix, df_cells, df_fovs["FOV"], MIN_CELLS
)
rule_presence_matrix = all_states
cell_matrix = cell_share_matrix(df_cells)
print(f"{len(rule_matrix)} rows; {all_states.shape[0]} rules × {all_states.shape[1]} FOVs")

## All-FOV comparisons

In [ ]:
organ_comparison = compare_groups(
    rule_presence_matrix, fovs_for("Colon"), fovs_for("Duodenum"), "Colon", "Duodenum"
)
stage_comparisons = {
    (organ, a, b): compare_groups(
        rule_presence_matrix, fovs_for(organ, a), fovs_for(organ, b), a, b
    )
    for organ in ORGANS for a, b in STAGE_PAIRS
}
organ_cells = compare_groups(cell_matrix, fovs_for("Colon"), fovs_for("Duodenum"), "Colon", "Duodenum")
stage_cells = {
    (organ, a, b): compare_groups(cell_matrix, fovs_for(organ, a), fovs_for(organ, b), a, b)
    for organ in ORGANS for a, b in STAGE_PAIRS
}
organ_metrics = mean_metrics_by_group(rule_matrix, df_fovs.set_index("FOV")["Organ"])
stage_metrics = mean_metrics_by_group(rule_matrix, df_fovs.set_index("FOV")[SCORE_COLUMN])

In [ ]:
dv.plot_volcano(organ_comparison, "Colon", "Duodenum", scope="all stages", min_gap=MIN_GAP)
dv.plot_dumbbell(organ_comparison, "Colon", "Duodenum", scope="all stages", min_gap=MIN_GAP)
display(report_table(organ_comparison, "Colon", "Duodenum", organ_metrics))

for (organ, a, b), result in stage_comparisons.items():
    dv.plot_volcano(result, a, b, scope=organ, min_gap=MIN_GAP)
    dv.plot_dumbbell(result, a, b, scope=organ, min_gap=MIN_GAP)
    dv.plot_dumbbell(stage_cells[(organ, a, b)], a, b, scope=organ, top_n=15,
                     value_label="share of the cells (%)", what="cell types")
    display(report_table(result, a, b, stage_metrics))

## All-FOV severity trends

In [ ]:
stage_trends, stage_cell_trends, summary = {}, {}, []
for organ in ORGANS:
    groups = {stage: fovs_for(organ, stage) for stage in STAGES}
    rule_trend = compare_stages(rule_presence_matrix, groups)
    cell_trend = compare_stages(cell_matrix, groups)
    stage_trends[organ], stage_cell_trends[organ] = rule_trend, cell_trend
    moved = rule_trend["fdr"] <= FDR_CUTOFF
    summary.append({
        "organ": organ, "rules tested": len(rule_trend),
        "moved": int(moved.sum()),
        "climbing": int((moved & (rule_trend["slope"] > 0)).sum()),
        "fading": int((moved & (rule_trend["slope"] < 0)).sum()),
        "steady": int((moved & rule_trend["steady"]).sum()),
    })
display(pd.DataFrame(summary).set_index("organ"))

for organ in ORGANS:
    for rising in (True, False):
        dv.plot_rules_with_cells(stage_trends[organ], stage_cell_trends[organ],
                                 STAGES, rising=rising, scope=organ)
    display(trend_table(stage_trends[organ]))
    dv.plot_trend(stage_cell_trends[organ], STAGES, scope=organ,
                  what="Cell types", value_label="share of the cells (%)")
    display(trend_table(stage_cell_trends[organ]))

## Eligibility-controlled comparisons

In [ ]:
if USE_CELL_ELIGIBILITY:
    scope = f"at least {MIN_CELLS} cells of both types"
    organ_eligible = eligible_compare(fovs_for("Colon"), fovs_for("Duodenum"), "Colon", "Duodenum")
    dv.plot_volcano(organ_eligible, "Colon", "Duodenum", scope=scope, min_gap=MIN_GAP)
    dv.plot_dumbbell(organ_eligible, "Colon", "Duodenum", scope=scope, min_gap=MIN_GAP)
    display(organ_eligible[organ_eligible["fdr"] <= FDR_CUTOFF].head(15).round(3))

    eligible_stage_comparisons = {}
    for organ in ORGANS:
        for a, b in STAGE_PAIRS:
            result = eligible_compare(fovs_for(organ, a), fovs_for(organ, b), a, b)
            eligible_stage_comparisons[(organ, a, b)] = result
            dv.plot_volcano(result, a, b, scope=f"{organ}; {scope}", min_gap=MIN_GAP)
            dv.plot_dumbbell(result, a, b, scope=f"{organ}; {scope}", min_gap=MIN_GAP)

## Eligibility-controlled severity trends

In [ ]:
if USE_CELL_ELIGIBILITY:
    eligible_stage_trends = {}
    for organ in ORGANS:
        groups = {stage: fovs_for(organ, stage) for stage in STAGES}
        ordered = eligible_trend(groups)
        eligible_stage_trends[organ] = ordered
        dv.plot_trend(ordered, STAGES, scope=f"{organ}; at least {MIN_CELLS} cells of both types",
                      top_n=TOP_TRENDS, value_label="net prevalence among eligible FOVs (%)")
        table = trend_table(ordered, TOP_TRENDS)
        for stage in STAGES:
            table[f"{stage} eligible n"] = ordered.reindex(table.index)[f"n_eligible_{stage}"]
        display(table)
        selected = strongest_trends(ordered)
        for rule in selected.index:
            dv.plot_state_breakdown(eligible_states, selected.loc[[rule]], groups, STAGES,
                                    scope=f"{organ}; at least {MIN_CELLS} cells of both types")

## Detailed severity screens

### Pathological score

In [ ]:
pathological_results = show_score_results("Pathological score") if USE_CELL_ELIGIBILITY else {}

### Clinical score

In [ ]:
clinical_results = show_score_results("Clinical score") if USE_CELL_ELIGIBILITY else {}

## Stage-matched organ screens

In [ ]:
organ_results = {score: show_organ_results(score) for score in ("Pathological score", "Clinical score")} if USE_CELL_ELIGIBILITY else {}